# Gap Stall and Attribution — Measuring "Which Branch Pushed Up the Dual Bound"

When the gap doesn't shrink, SCIP's log only tells you the "node count" and "incumbent gap".
`minlpkit.collectors.attribution` takes a snapshot of the global dual bound at each branch, and quantifies "which branches were effective" by **attributing the dual bound increment to the immediately preceding branch (variable and type)**. It also performs **automatic detection of stalled intervals** (time periods where the improvement rate is slower than average).

1. **What to Observe** — Dual bound snapshots per `NODEBRANCHED` and how `Δdual` is attributed
2. **Visualization** — Dual bound transition + stacked contributions by type + shaded stalled intervals
3. **Interpretation** — Why stalls are detected as "slower than average" rather than "flat", and correspondence with the `dual_stall` diagnosis
4. **Summary**

The subject is **Batch Reactor Scheduling** (`samples/others/scheduling_plant.py`), a known hard problem (`.claude/skills/minlp-run/SKILL.md`: stalls at a 72.5% gap in 300 seconds).

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for sub in ["samples", "samples/others"]:
    p = str(ROOT / sub)
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import HTML, display

def show(fig):
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False,
                             config={"displayModeBar": False})))

import minlpkit as mk
import scheduling_plant as sp
from minlpkit.collectors.attribution import (
    AttributionCollector, attribute_gains, gain_by_kind, gain_by_variable,
    detect_stalls, solve_and_attribute)

C = dict(ink="#0b0b0b", ink2="#52514e", muted="#898781", grid="#e1e0d9",
         axis="#c3c2b7", surface="#fcfcfb", s1="#2a78d6", s2="#008300", warn="#c25a00")
KIND_COLOR = {"spatial": C["s1"], "integer": C["s2"], "binary": "#e87ba4", "root": C["muted"]}
KIND_LABEL = {"spatial": "空間分枝(連続)", "integer": "整数分枝", "binary": "0-1分枝", "root": "根"}
FONT = 'system-ui, -apple-system, "Segoe UI", sans-serif'
print("repo root:", ROOT)

## 1. What to Observe — Dual Bound Snapshots and Attribution per Branch

`AttributionCollector` records a row `(elapsed time, node count, global dual bound, branching variable, kind)` every time `NODEBRANCHED` occurs. `BESTSOLFOUND` is used separately to measure TTFF (seconds to the first feasible solution).

`attribute_gains` sorts the records chronologically and calculates `Δdual = max(0, dual[i+1] - dual[i])`, assuming that **the branch at record i contributed to pushing up the global dual bound from i to i+1**. This is an approximate credit assignment, but it is sufficient to grasp the trend of "whether spatial (spatial branching on continuous variables) or discrete (integer/0-1 branching) is pushing the dual bound".

In [ ]:
col = AttributionCollector()
m = sp.build_model()
m.includeEventhdlr(col, "AttributionCollector", "attributes dual gains")
m.setParam("timing/clocktype", 2)
m.setParam("limits/time", 60)
m.hideOutput()
m.optimize()
raw = col.to_frame()
d = attribute_gains(raw)
print(f"分枝レコード数: {len(raw)}  求解時間: {m.getSolvingTime():.1f}s  "
     f"最終gap: {m.getGap():.1%}  TTFF: {col.first_sol_time:.2f}s")
d[["time", "nodes", "dual", "branch_var", "kind", "dual_gain"]].head(6)

## 2. Visualization — Dual Bound Transition + Contribution by Type + Stalled Intervals

`detect_stalls` places time on evenly spaced grids, interpolates the dual bound using the cumulative maximum, and then extracts intervals where **the improvement per cell is less than half the overall average cell improvement** as "stalls"
(because judging purely by "flatness" overlooks cases where it is actually improving continuously but very slowly).

In [ ]:
stalls = detect_stalls(d)
print(f"検出された停滞区間: {len(stalls)}個")
for a, b in stalls:
    print(f"  {a:.1f}s - {b:.1f}s (幅{b-a:.1f}s)")

fig = go.Figure()
for a, b in stalls:
    fig.add_vrect(x0=a, x1=b, fillcolor=C["muted"], opacity=0.15, line_width=0)
dd = d.dropna(subset=["dual"])
fig.add_trace(go.Scatter(x=dd["time"], y=dd["dual"], mode="lines",
    line=dict(color=C["s2"], width=2, shape="hv"), name="大域双対境界"))
for kind in ["spatial", "integer", "binary"]:
    kd = d[(d["kind"] == kind) & (d["dual_gain"] > 0)]
    if kd.empty:
        continue
    fig.add_trace(go.Scatter(x=kd["time"], y=kd["dual"], mode="markers",
        marker=dict(color=KIND_COLOR[kind], size=6), name=f"改善: {KIND_LABEL[kind]}"))
fig.update_layout(
    title=dict(text="双対境界推移(灰帯=改善が平均の半分未満の停滞区間)",
              font=dict(color=C["ink"], size=15, family=FONT), x=0.01),
    paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
    font=dict(family=FONT, color=C["ink2"], size=12),
    xaxis=dict(title="経過時間[s]", gridcolor=C["grid"], linecolor=C["axis"], tickfont=dict(color=C["muted"])),
    yaxis=dict(title="大域双対境界", gridcolor=C["grid"], linecolor=C["axis"], tickfont=dict(color=C["muted"])),
    margin=dict(l=60, r=20, t=48, b=44), height=420,
    legend=dict(orientation="h", y=1.0, yanchor="bottom", x=1.0, xanchor="right", font=dict(size=11)))
show(fig)

In [ ]:
gk = gain_by_kind(d)
total = gk["dual_gain"].sum()
fig = go.Figure(go.Bar(
    x=gk["kind"].map(lambda k: KIND_LABEL.get(k, k)), y=gk["dual_gain"],
    marker=dict(color=[KIND_COLOR.get(k, C["muted"]) for k in gk["kind"]]),
    text=[f"{v/total:.0%}" for v in gk["dual_gain"]], textposition="outside"))
fig.update_layout(
    title=dict(text="双対境界改善の型別内訳(帰属合計)",
              font=dict(color=C["ink"], size=14, family=FONT), x=0.01),
    paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
    font=dict(family=FONT, color=C["ink2"], size=12),
    yaxis=dict(title="Δdual 合計", gridcolor=C["grid"], linecolor=C["axis"], tickfont=dict(color=C["muted"])),
    margin=dict(l=60, r=20, t=44, b=40), height=300)
show(fig)

gv = gain_by_variable(d, top=10).iloc[::-1]
fig2 = go.Figure(go.Bar(x=gv["dual_gain"], y=gv["branch_var"], orientation="h",
    marker=dict(color=[KIND_COLOR.get(k, C["muted"]) for k in gv["kind"]]),
    text=[f"{v:.1f}" for v in gv["dual_gain"]], textposition="outside"))
fig2.update_layout(
    title=dict(text="双対境界改善への寄与が大きい変数トップ10",
              font=dict(color=C["ink"], size=14, family=FONT), x=0.01),
    paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
    font=dict(family=FONT, color=C["ink2"], size=12),
    xaxis=dict(title="Δdual 合計", gridcolor=C["grid"], linecolor=C["axis"], tickfont=dict(color=C["muted"])),
    margin=dict(l=110, r=40, t=44, b=40), height=340)
show(fig2)
print(f"spatial寄与 = {gk.loc[gk['kind']=='spatial','dual_gain'].sum()/total:.1%}")

## 3. Interpretation — Correspondence Between Stalls and the `dual_stall` Diagnostic Rule

The `dual_stall` rule fires when `n_stalls>=1` and `gap>=0.05`, recommending "addition of valid inequalities, tightening variable bounds, and Big-M elimination." However, the recommendation only tells you that "the symptom exists", so verifying whether it actually works via `mk.compare_variants` before/after is essential (refer to the respective pages in the Methods Guide).

In [ ]:
report = mk.analyze(sp.build_model, name="plant-attribution", time_limit=15)
print(report.summary())
for f in report.findings:
    if f["id"] in ("dual_stall", "weak_relaxation"):
        print(f"\n[{f['severity']}] {f['id']}: {f['symptom']}")
        print("  根拠:", f["evidence"])
        print("  手順:", f["recipe"])

## Summary

- The attribution collector is a tool that attributes "when and by what the dual bound was improved" at the branch level, opening the black box of the branch-and-bound method. Automatic stall interval detection correctly captures cases of continuous, slow improvement by setting the criterion to "slower than average" rather than strictly "flat".
- The breakdown of `gain_by_kind` serves as criteria for both `weak_relaxation` (if spatial branching dominates, non-convex relaxation is the bottleneck) and `decomposable`/`dual_stall` — one collector supports multiple diagnostic rules.
- Even if stalled intervals or contributing variables are identified, they are not "solutions" in themselves. The next step is to measure what actually works on the corresponding page of the [Methods Guide](../../playbook/index.md) (linked from each page to its notebook).

Related: [Methods Guide 0. Diagnosis Itself](../../playbook/00-diagnose.md) /
[Notebook 1: McCormick + Spatial Branching Tree](01_mccormick_spatial_tree.ipynb) /
API [`minlpkit.collectors`](../../api/pipeline.md).